## 1. Create Dataset

In [1]:
import numpy as np

def one_hot(val, size):
    vec = [0] * size
    vec[val] = 1
    return vec
              
# Create dataset 
X = np.empty((1, 4), dtype=int)
number = 1
for context in range(2,4):
    for a1 in range(3):
        
        # If mouse goes to cue it sees Right or Left
        if a1 == 2:
            o2 = context
            
        # If it goes Left  or Right it sees Cheese or Shock
        elif a1 == 1:
            o2 = 3 - context
            
        else:
            o2 = context - 2
            
        for a2 in range(3):
            if a1 != 2: # TRAP
                o3 = o2
            else:
                if a2 == 2:
                    o3 = context
            
                # If it goes Left  or Right it sees Cheese or Shock
                elif a2 == 1:
                    o3 = 3 - context
                    
                else:
                    o3 = context - 2
                    
            # print(f"number{number} a1:{a1} o2:{o2} a2:{a2} o3:{o3}")
            number += 1
            datapoint = np.array([a1, o2, a2, o3])

            X = np.vstack((X, datapoint))

  
X = X[1:] # Remove the first empty row          
print(X.shape) # 18 possible paths and 4 variables (a1, o2, a2, o3)


(18, 4)


## 2. Initialize Tensor Net

In [2]:
from tn4ml.models.mps import MatrixProductState
from tn4ml.metrics import NegLogLikelihood
import numpy as np
import optax
from tn4ml.util import EarlyStopping 
import jax.numpy as jnp

def identity_3d_cubical(shape, backend='numpy'):
    """
    Creates a 3D identity-like tensor where only elements with i == j == k are 1.

    Args:
        shape (tuple): Shape of the 3D tensor (e.g., (3, 3, 3)).
        backend (str): 'numpy' or 'jax'. Defaults to 'numpy'.

    Returns:
        ndarray: 3D identity-like tensor.

    Example:
        >>> identity_3d_cubical((3, 3, 3))
        array([[[1, 0, 0],
                [0, 0, 0],
                [0, 0, 0]],
               [[0, 0, 0],
                [0, 1, 0],
                [0, 0, 0]],
               [[0, 0, 0],
                [0, 0, 0],
                [0, 0, 1]]])
    """
    if backend == 'jax':
        xp = jnp
    else:
        xp = np

    def _identity_rule(i, j, k):
        return xp.where(i == j, 1, 0)

    return xp.fromfunction(
        xp.vectorize(_identity_rule, otypes=[float]),
        shape,
        dtype=float
    )
    
# Initialize MPS
max_bond_dim = 16

A1 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4)) # Should be 3, but in order to pass one Embed function, actions will have a 4th null state.
O2 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4))
A2 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4)) # Idem
O3 = identity_3d_cubical((max_bond_dim, max_bond_dim, 4))
tensors = [A1, O2, A2, O3]

model = MatrixProductState(tensors)



c:\Users\darth\Documents\VSCode\tn4mlTmaze\myenv\Lib\site-packages\tn4ml\models\smpo.py:8: FutureWarning: The module 'quimb.tensor.tensor_1d' is deprecated and will be removed in a future release. Most functionality can be still be accessed directly from 'quimb.tensor' instead. The actual implementations have moved to `quimb.tensor.tn1d.core`.
  from quimb.tensor.tensor_1d import MatrixProductState, TensorNetwork1DOperator, TensorNetwork1DFlat


## 3. Embed data with a one hot feature map

In [3]:


from tn4ml.embeddings import Embedding

class OneHotEmbedding(Embedding):
    """One-hot feature map."""
    def __init__(self, num_states: int, **kwargs):
        assert num_states >= 1
        self.num_states = num_states
        super().__init__(**kwargs)

    @property
    def dim(self) -> int:
        return self.num_states

    @property
    def input_dim(self) -> int:
        return 1

    def __call__(self, x: int) -> jnp.ndarray:
        x_int = np.round(x).astype(int)
        one_hot = jnp.zeros(self.num_states)
        one_hot = one_hot.at[x_int].set(1.0)
        return one_hot

embedding = OneHotEmbedding(num_states=4)



learning_rate = 1e-4
earlystop = EarlyStopping(min_delta=0, patience=10, monitor='loss', mode='min')
device = 'cpu'

# define training parameters
epochs = 100
batch_size = 18
optimizer = optax.adam
strategy = 'global'
loss = NegLogLikelihood
train_type = 0 # TrainingType.UNSUPERVISED

model.configure(optimizer=optimizer, strategy=strategy, loss=loss, train_type=train_type, learning_rate=learning_rate, device=device)

## 4. Train TN

In [4]:

history = model.train(X,
            epochs=epochs,
            batch_size=batch_size,
            embedding = embedding,
            normalize = True,
            dtype = jnp.float64,
            earlystop = earlystop,
            cache = True,
            )

c:\Users\darth\Documents\VSCode\tn4mlTmaze\myenv\Lib\site-packages\tn4ml\models\model.py:303: UserWarning: Explicitly requested dtype float64 requested in ones is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  dummy_input = jnp.ones(shape=input_shape, dtype=inputs_dtype)
epoch:   0%|          | 0/100 [00:00<?, ?it/s]c:\Users\darth\Documents\VSCode\tn4mlTmaze\myenv\Lib\site-packages\tn4ml\models\model.py:946: UserWarning: Explicitly requested dtype float64 requested in asarray is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  x = jax.numpy.asarray(x, dtype=dtype)
epoch:   1%|           1/100 , loss=-5.5452   

Waiting for 1 epochs.


epoch:   1%|           1/100 , loss=5.5451 

Waiting for 2 epochs.


epoch:   2%|▏          2/100 , loss=5.5447

...

epoch:   8%|▊          8/100 , loss=5.5424

Waiting for 9 epochs.


epoch:   9%|▉          9/100 , loss=5.5420

Training stopped by EarlyStopping on epoch: 0


epoch:  11%|█          11/100 , loss=5.5416


## 4. Evaluate TN

In [5]:
model.partial_trace_to_mpo([0, 1])

MatrixProductOperator(tensors=2, indices=5, L=2, max_bond=65536)